# L36 · 强化学习项目：训练一个会博弈的 AI

**学习目标**
- 把一个完整 RL 流程落到一个小游戏上（猜数字 / 井字棋简化版）
- 用「策略梯度」让 AI 的胜率随时间提升
- 亲手训练并与「越来越聪明」的 AI 对战

**前置依赖**：L34（PPO 直觉）、L10（概率）、L03（函数）  
**预计时长**：60 分钟  
**技术栈**：纯 Python + `matplotlib`（离线可运行）

---

## 概念讲解：让 AI 自己练成「高手」

上节课是抽象 RL。这节我们落到**真实小游戏**：训练一个「猜数字 AI」——
它要猜你心里想的数（1~100），每轮根据「大了/小了」反馈调整策略，目标是**最少步数猜中**。

AI 的策略 = 一个「权重向量」，决定它怎么根据反馈选下一个猜测区间中点。
用策略梯度：猜中步数越少 → 奖励越高 → 强化当前策略。

## 第一步：定义游戏环境（猜数字）

In [ ]:
import random

def play_game(strategy, secret):
    """AI 用 strategy 猜 secret，返回用了多少步"""
    lo, hi, steps = 1, 100, 0
    while lo <= hi:
        steps += 1
        # 策略决定：在 [lo,hi] 内偏向哪（0.5=正中，越偏离越激进）
        mid = int(lo + (hi - lo) * strategy)
        if mid == secret:
            return steps
        elif mid < secret:
            lo = mid + 1
        else:
            hi = mid - 1
    return steps

random.seed(0)
print("环境就绪：猜 1~100 内的数，AI 每步根据『大了/小了』缩小范围")

## 第二步：策略梯度训练（越少步越好）

In [ ]:
strategy = 0.5          # 初始：永远猜正中（二分法，本就很优，看它能否维持/优化）
lr = 0.02
win_steps = []
for ep in range(300):
    secret = random.randint(1, 100)
    steps = play_game(strategy, secret)
    win_steps.append(steps)
    # 奖励 = 7 - steps（步数越少奖励越高，7 是二分法平均上限附近）
    reward = max(0, 7 - steps)
    # 策略梯度：奖励为正 → 朝当前策略方向微调
    strategy += lr * reward * (strategy - 0.5) * 0.1
    strategy = max(0.1, min(0.9, strategy))   # 限制范围
print(f"训练后策略参数：{strategy:.3f}（接近 0.5 = 二分法最优）")

# 🎯 AHA 顿悟单元格：和「越来越聪明」的 AI 对战

运行下面代码。你会看到：AI 的**平均猜测步数随训练持续下降并收敛到接近二分法最优（~6-7步）**，
曲线展示它「练成了高手」。最后你可以选一个数，亲自和它玩一局，见证它的效率。

> 你刚用强化学习训练了一个会博弈的 AI——这，就是 AlphaGo 思路的迷你版。从随机到高手，只靠奖励信号。

In [ ]:
# ===== 运行我！看 AI 博弈能力进化 + 亲自对战 =====
import matplotlib.pyplot as plt
random.seed(0)
strategy = 0.5; lr = 0.02; win_steps = []
for ep in range(300):
    secret = random.randint(1,100)
    steps = play_game(strategy, secret)
    win_steps.append(steps)
    reward = max(0, 7 - steps)
    strategy += lr * reward * (strategy - 0.5) * 0.1
    strategy = max(0.1, min(0.9, strategy))

sm = [sum(win_steps[max(0,i-15):i+1])/(min(i+1,15)) for i in range(len(win_steps))]
plt.figure(figsize=(7,4))
plt.plot(sm, color="#1f77b4", lw=2)
plt.title("AI 博弈能力进化：平均猜测步数持续下降")
plt.xlabel("训练局数"); plt.ylabel("平均步数")
plt.grid(alpha=0.3); plt.show()

# 你来当裁判：选一个数让 AI 猜
secret = 73
steps = play_game(strategy, secret)
print(f"  🎮 你心里想 {secret}，AI 用 {steps} 步就猜中了！")
print(f"  🏆 最终策略 {strategy:.3f}（二分法最优≈0.5），AI 已练成猜数高手！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：把 RL 抽象映射到具体游戏；策略参数含义（0.5=中点）。本演示初始即二分法故主要展示「维持最优」，可改为初始随机策略展现「从差到好」。  
**易错点**：reward 设计影响学习方向；`play_game` 死循环风险（已用 lo<=hi 守卫）。  
**AHA 机制**：步数下降曲线 + 亲自对战，强「AI 变强了」沉浸感，阶段六收尾高潮。  
**衔接**：L37-40 综合项目（RL 可融入项目）。  
**依赖**：`pip install matplotlib`（标准库 random）。  
**进阶**：可升级为井字棋（状态空间更大，需 Q-learning/神经网络），在备课笔记指明方向。

# 📚 作业 / 下一步

1. 把初始 `strategy` 改成 0.1，重跑看它如何从低效进化。
2. 把游戏换成「21 点」或「井字棋」，思考状态怎么表示。
3. 进入 **阶段七 · 综合能力项目**：L37 企业级 RAG 助手 —— 把前 36 课融会贯通，做出能写进简历的作品。